# GoalLab Stage 11: Tool Gateway Integration

Created by Shivam Bharadwaj · [Course home](../../../README.md) · [Stage map](../STAGES.md)

[Stage 10](10_verifiable_reasoning_grpo.ipynb) · [Stage 12](12_sampling_and_selection.ipynb)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bshivambharadwaj/Reinforcement-Learning-Course/blob/main/projects/goallab/apply-theory/11_tool_gateway_integration.ipynb)

**Build:** integrate the learned routing policy with the shared workspace gateway, final evaluator, and persisted evidence record. This is GoalLab Stage 11: Tool Gateway Integration.

**Run:** use the full repository, install `requirements.txt` (Stage 09 also uses `requirements-modern.txt`), then restart the kernel and run all cells. Every stage reconstructs its inputs from shared GoalLab modules; earlier notebook execution is not required. The project progression is cumulative in capabilities, without hidden notebook state.

Code: MIT. Text and plots: CC BY 4.0. Results below are reproduced from the supplied GoalLab cases; the evaluation section explains their scope and failure cases.

**Learn the algorithm first:** [Course Notebook 11](../../../notebooks/11_tool_agent_lab.ipynb). Then use this GoalLab stage to apply it to evidence gathering and briefing verification.


In [1]:
import sys, subprocess, platform, tempfile, json
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rl_course').is_dir()), Path.cwd())
if IN_COLAB and not (ROOT / 'rl_course').exists():
    ROOT = Path('/content/Reinforcement-Learning-Course')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/bshivambharadwaj/Reinforcement-Learning-Course.git', str(ROOT)], check=True)
if not (ROOT / 'rl_course' / 'goallab.py').exists():
    raise FileNotFoundError('Use the complete repository revision containing the GoalLab stages.')
sys.path.insert(0, str(ROOT))
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Markdown, display
from rl_course import goallab as goal
from rl_course import goallab_learning as learning
torch.set_num_threads(2)
%matplotlib inline
INK, GOLD, SAGE, PLUM, PAPER = '#292332', '#af824a', '#768165', '#87708c', '#f5f1e8'
plt.rcParams.update({'figure.figsize': (9, 4), 'figure.dpi': 110, 'figure.facecolor': PAPER,
    'axes.facecolor': PAPER, 'text.color': INK, 'axes.labelcolor': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=[GOLD, SAGE, PLUM, INK]), 'axes.grid': True, 'grid.alpha': .15})
def table(headers, rows):
    display(Markdown('| ' + ' | '.join(headers) + ' |\n|' + '|'.join(['---'] * len(headers)) + '|\n' +
        '\n'.join('| ' + ' | '.join(str(x).replace('|', '&#124;').replace('\n', '<br>') for x in row) + ' |' for row in rows)))
def learning_plot(runs):
    for label, run in runs.items():
        h = run['history']
        plt.plot(h[:, 0], h[:, 1], label=label)
    plt.xlabel('Training environment actions'); plt.ylabel('Exact policy return')
    plt.legend(); plt.tight_layout(); plt.show()
def report_policies(runs):
    table(['Controller', 'Exact return', 'Held-out briefing success', 'Calls/case'],
          [[label, round(run['env'].values(run['policy'])[0][run['env'].start_index], 3),
            f"{learning.heldout(run['policy'])['success']:.1%}", learning.heldout(run['policy'])['calls']]
           for label, run in runs.items()])
workspace = goal.make_workspace(7, 'test')
print(f'{goal.VERSION} | Python {platform.python_version()} | NumPy {np.__version__} | Torch {torch.__version__} | CPU')

GoalLab-v1 | Python 3.12.4 | NumPy 1.26.4 | Torch 2.11.0+cu130 | CPU


## Shared contract

One source is an approved measurement for the requested period. Other sources are drafts or forecasts, deliberately newer and numerically different. Training rows use 0..3; held-out rows use 4..7 and a different period. Authority rules stay the same. `search` reveals IDs and timestamps; `read` reveals content. A valid briefing needs the correct value, period, and citation.

The simulator and models are readable teaching code, not a security boundary. Final `verify()` is outside the policy interface. The first eight stages use a finite two-source abstraction: read twice, choose a citation, then submit or claim completion. Claiming completion produces no briefing. This fixed horizon makes exact comparisons possible; later stages vary investigation budgets.

## Learning objectives and the stage contract

Implement the gateway contract; distinguish validation from evaluation; trace failed and accepted submissions.

**Input:** the shared evidence workspace or the preceding component reconstructed below. **Output:** the capability named in this stage, evaluated against the common briefing contract. Read the worked calculation before running the full experiment; then inspect the implementation and predict the constraint exercise.

## Work it out: validation and verification answer different questions

The gateway asks whether a tool request is permitted. The evaluator asks whether the delivered result is correct. A read citation may still be a draft, and a permitted submission may still contain a wrong number.

Run three cases below: unread citation, read draft, and read approved source. Inspect acceptance separately from verified success. This boundary is more important than whether the controller is tabular, neural, or language-based.

In [2]:
w=goal.make_workspace(7,'test')
rows=[]
for scenario in ['Unread citation','Read draft','Read approved']:
    source_index=next(i for i,s in enumerate(w.sources) if goal.usable(s,w.period)) if scenario=='Read approved' else next(i for i,s in enumerate(w.sources) if not goal.usable(s,w.period))
    session=goal.Session(w,3); source=w.sources[source_index]
    if scenario!='Unread citation': session.execute('read',source_index)
    accepted=session.execute('submit',goal.Briefing(source.value,source.source_id,w.period))['ok']
    rows.append([scenario,accepted,goal.verify(w,session.artifact)['success']])
table(['Scenario','Gateway accepted','Independent success'],rows)
assert rows==[['Unread citation',False,False],['Read draft',True,False],['Read approved',True,True]]

| Scenario | Gateway accepted | Independent success |
|---|---|---|
| Unread citation | False | False |
| Read draft | True | False |
| Read approved | True | True |

## Build the component

The central implementation is included here so you can step through and edit the update or tool logic. The shared modules retain the same reference implementation for the integrated project. Inspect shapes, terminal handling, frozen quantities, and the evaluator boundary before training.

In [3]:
from dataclasses import asdict
from copy import deepcopy
Briefing=goal.Briefing
class Session:
    """Isolated tool state with a hard call budget and replayable event records.

    Python objects are inspectable in these lessons. This is a logical boundary,
    not an OS sandbox. External services and private accounts are not connected.
    """
    def __init__(self, workspace, budget=6):
        if not isinstance(budget, int) or budget < 0:
            raise ValueError('budget must be a nonnegative integer')
        self.workspace = workspace
        self.budget = budget
        self.events = []
        self.reads = {}
        self.artifact = None
        self.closed = False

    def execute(self, action, argument=None):
        if self.closed or len(self.events) >= self.budget:
            raise RuntimeError('Session closed or budget exhausted')
        result = {'ok': False}
        if action == 'search':
            result = {'ok': True, 'sources': self.workspace.search()}
        elif action == 'read' and isinstance(argument, int) and 0 <= argument < len(self.workspace.sources):
            source = self.workspace.sources[argument]
            self.reads[source.source_id] = source
            result = {'ok': True, 'source': asdict(source)}
        elif action == 'submit' and isinstance(argument, Briefing):
            # Enforce citation-read permission without grading correctness.
            if argument.abstained or argument.citation in self.reads:
                self.artifact = argument
                self.closed = True
                result = {'ok': True, 'submitted': True}
        elif action == 'claim_done':
            result = {'ok': True, 'message': 'Claim recorded; no artifact submitted'}
        self.events.append({'action': action,
                            'argument': asdict(argument) if isinstance(argument, Briefing) else argument,
                            'result': result})
        return result

    def snapshot(self):
        return deepcopy(self)

## 1. Train against two evaluators

A verified reward requires a supported briefing. A flawed reward gives 1.5 for claiming completion without submitting anything. The tools and held-out evaluator are identical. A dispatcher rewarded for saying “done” may learn exactly that shortcut.

In [4]:
runs={'Verified':learning.train_tabular('Q-learning',7,1200),
      'Claim-reward exploit':learning.train_tabular('Q-learning',7,1200,goal.EvidenceMDP(naive=True))}
report_policies(runs)
for label,r in runs.items():
    session=r['env'].deploy(r['policy'],workspace)
    print(label,session.artifact,goal.verify(workspace,session.artifact))

| Controller | Exact return | Held-out briefing success | Calls/case |
|---|---|---|---|
| Verified | 0.92 | 100.0% | 4.0 |
| Claim-reward exploit | 1.42 | 0.0% | 4.0 |

Verified Briefing(value=11, citation='test-7-source-1', period='2026-02', abstained=False) {'success': True, 'supported': True, 'value_correct': True}
Claim-reward exploit None {'success': False, 'supported': False, 'value_correct': False}


## 2. Enforce the gateway contract and save evidence

The gateway refuses a citation to an unread source. This is enforced independently of reward. It still cannot establish that every submitted number is correct; the evaluator supplies that check after submission.

The saved record includes case/version, event trace, artifact, budget, and verification. Files go to a temporary directory rather than modifying the repository.

In [5]:
session=Session(workspace,4)
source=workspace.sources[0]
denied=session.execute('submit',goal.Briefing(source.value,source.source_id,workspace.period))
assert not denied['ok']
good=runs['Verified']['env'].deploy(runs['Verified']['policy'],workspace)
with tempfile.TemporaryDirectory(prefix='goallab-stage11-') as output:
    record=goal.save_record(good,Path(output)/'briefing.json')
    print(json.dumps(record,indent=2))

{
  "version": "GoalLab-v1",
  "case": "test-7",
  "budget": 4,
  "events": [
    {
      "action": "read",
      "argument": 0,
      "result": {
        "ok": true,
        "source": {
          "source_id": "test-7-source-0",
          "status": "draft",
          "kind": "forecast",
          "period": "2026-02",
          "updated": 2,
          "rows": [
            16,
            5
          ]
        }
      }
    },
    {
      "action": "read",
      "argument": 1,
      "result": {
        "ok": true,
        "source": {
          "source_id": "test-7-source-1",
          "status": "approved",
          "kind": "measurement",
          "period": "2026-02",
          "updated": 1,
          "rows": [
            6,
            5
          ]
        }
      }
    },
    {
      "action": "choose_citation",
      "argument": 1,
      "result": {
        "ok": true
      }
    },
    {
      "action": "submit",
      "argument": {
        "value": 11,
        "citation": "test-

## Interpret the evidence

A permitted request can still produce a false briefing. Keep gateway acceptance and semantic evaluation as separate fields in the record. The in-process gateway is inspectable teaching code, not hardened isolation.

## Change a constraint: predict first

Cut the tool allowance to one call. Can a session read and submit? Explain why a reward penalty is insufficient to enforce the limit.

### Worked solution

Run this after writing your prediction. Compare the changed condition with the reference condition above.

In [6]:
limited=Session(workspace,1); limited.execute('read',0)
try:
    limited.execute('claim_done')
except RuntimeError as error:
    print('Enforced:',error)
assert len(limited.events)==1
print('A failed over-budget request cannot silently increase the allowance.')

Enforced: Session closed or budget exhausted
A failed over-budget request cannot silently increase the allowance.


## What this stage contributes

A complete reference tool loop, learned controller, and reviewable artifact. Stages 12–14 now improve how computation is spent before the final submission. Theory: [agents](../../../chapters/40-multimodal-rl-and-rl-for-ai-agents/README.md).

[Stage 10](10_verifiable_reasoning_grpo.ipynb) · [Stage 12](12_sampling_and_selection.ipynb)